In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

import minedojo
from minedojo.sim import InventoryItem

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor, VecVideoRecorder
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

import torch
import torch.nn as nn



[INFO:minedojo.tasks] Loaded 1581 Programmatic tasks, 1560 Creative tasks, and 1 special task: "Playthrough". Totally 3142 tasks loaded.


In [2]:
SEED = "OhGODHELPME"

BASE_CFG = dict(
    task_id="open-ended",
    image_size=(160, 256),
    initial_inventory=[
        InventoryItem(slot=0, name="obsidian", variant=None, quantity=64),
        InventoryItem(slot=1, name="flint_and_steel", variant=None, quantity=1),
    ],
    world_seed=SEED,
    use_voxel=True,
    voxel_size=dict(xmin=-4, ymin=-4, zmin=-4, xmax=4, ymax=4, zmax=4),
)

# Voxel grid shape derived from cfg: (x_range, y_range, z_range)
# xmin=-4..xmax=4  → 9,  ymin=-4..ymax=4 → 9,  zmin=-4..zmax=4 → 9
VOXEL_SHAPE = (9, 9, 9)

In [3]:
def _has_portal_frame(obsidian_grid: np.ndarray) -> bool:
    """
    Checks whether a 4-wide × 5-tall rectangular obsidian frame exists
    in any vertical slice of the local voxel grid.
    obsidian_grid: bool array of shape VOXEL_SHAPE (x, y, z)
    """
    # Search over z-slices (treat z as depth, x as width, y as height)
    for z in range(obsidian_grid.shape[2]):
        plane = obsidian_grid[:, :, z]  # shape (9, 9)
        # Slide a 4-wide × 5-tall window
        for x in range(plane.shape[0] - 3):
            for y in range(plane.shape[1] - 4):
                frame = plane[x : x + 4, y : y + 5]
                # Frame = perimeter filled, interior hollow
                interior = frame[1:3, 1:4]
                perimeter_ok = (
                    frame[0, :].all()       # bottom row
                    and frame[3, :].all()   # top row
                    and frame[:, 0].all()   # left col
                    and frame[:, 4].all()   # right col
                )
                interior_ok = not interior.any()
                if perimeter_ok and interior_ok:
                    return True
    return False


In [4]:
class ObsidianPortalEnv(gym.Env):
    """
    Wraps a MineDojo open-ended environment.

    Observation space:
        - "rgb":    (C, H, W) uint8 image from the first-person camera
        - "voxel":  (9,9,9)   float32  — 1.0 where obsidian, else 0.0

    Action space: inherited directly from MineDojo (MultiDiscrete).

    Reward:
        +1.0  per *new* obsidian voxel placed since last step
        -0.01 time penalty per step
        +5.0  bonus when a portal frame is first detected
    """

    metadata = {"render_modes": ["rgb_array"]}

    def __init__(self, cfg: dict = BASE_CFG):
        super().__init__()
        self._env = minedojo.make(**cfg)

        # ---- observation space ----
        H, W = cfg["image_size"]
        self.observation_space = spaces.Dict(
            {
                "rgb": spaces.Box(low=0, high=255, shape=(3, H, W), dtype=np.uint8),
                "obsidian_mask": spaces.Box(
                    low=0.0, high=1.0, shape=VOXEL_SHAPE, dtype=np.float32
                ),
            }
        )

        # ---- action space (pass-through) ----
        self.action_space = self._env.action_space

        # ---- internal state ----
        self._prev_obsidian_count: int = 0
        self._portal_bonus_given: bool = False

    # ------------------------------------------------------------------
    def _extract_obs(self, raw_obs: dict) -> dict:
        """Convert MineDojo obs dict → our slim obs dict."""
        rgb = raw_obs["rgb"]  # already (3, H, W) uint8

        # MineDojo flattens voxel data into the top-level obs dict.
        # The block-name array is keyed as "voxels" (with an 's'), not "voxel".
        # Run smoke_test() with DEBUG=True to print all keys if this breaks.
        block_names: np.ndarray = raw_obs["voxels"]["block_name"]
        obsidian_mask = (block_names == "obsidian").astype(np.float32)

        # Cache masks so masked_action() can read them after each step.
        self._last_masks = raw_obs.get("masks", {})

        return {"rgb": rgb, "obsidian_mask": obsidian_mask}

    # ------------------------------------------------------------------
    # Action space indices  MultiDiscrete([3, 3, 4, 25, 25, 8, 244, 36])
    _IDX_FORWARD   = 0   # 1=forward, 2=back
    _IDX_STRAFE    = 1   # 1=left, 2=right
    _IDX_JUMP      = 2   # 1=jump, 3=sprint
    _IDX_PITCH     = 3   # 0-24 camera pitch delta
    _IDX_YAW       = 4   # 0-24 camera yaw delta
    _IDX_FUNC      = 5   # 0=noop 1=use 2=drop 3=attack 4=craft 5=equip 6=place 7=destroy
    _IDX_CRAFT_ARG = 6   # item index for craft
    _IDX_SLOT_ARG  = 7   # inventory slot for equip/place/destroy

    _FUNC_NOOP    = 0
    _FUNC_PLACE   = 6
    _FUNC_DESTROY = 7

    def masked_action(self, model_action: np.ndarray) -> np.ndarray:
        """
        Enforces MineDojo action masks on a raw PPO action so every step is
        either a valid movement or a valid place action.

        Mask sources (from obs["masks"]):
          action_type  (8,)  bool -- which functional actions are valid
          place        (36,) bool -- which inventory slots are placeable

        Invalid functional actions fall back to noop.
        Invalid place slots fall back to the first valid placeable slot,
        or noop if nothing is placeable.
        Pure noop (no movement, no functional action) forces forward=1.
        """
        action = model_action.copy()
        masks  = self._last_masks

        # 1. Validate functional action
        func = int(action[self._IDX_FUNC])
        action_type_mask = masks.get("action_type", np.ones(8, dtype=bool))
        if not action_type_mask[func]:
            func = self._FUNC_NOOP
            action[self._IDX_FUNC] = func

        # 2. Validate place slot if placing
        if func == self._FUNC_PLACE:
            place_mask = masks.get("place", np.zeros(36, dtype=bool))
            slot = int(action[self._IDX_SLOT_ARG])
            if not place_mask[slot]:
                valid_slots = np.where(place_mask)[0]
                if len(valid_slots) > 0:
                    action[self._IDX_SLOT_ARG] = int(valid_slots[0])
                else:
                    # Nothing placeable -- revert to noop
                    action[self._IDX_FUNC]     = self._FUNC_NOOP
                    action[self._IDX_SLOT_ARG] = 0

        # 2b. Validate destroy slot -- destroy mask is only true for non-empty
        #     slots; attempting to destroy air raises a ValueError in MineDojo.
        elif func == self._FUNC_DESTROY:
            destroy_mask = masks.get("destroy", np.zeros(36, dtype=bool))
            slot = int(action[self._IDX_SLOT_ARG])
            if not destroy_mask[slot]:
                valid_slots = np.where(destroy_mask)[0]
                if len(valid_slots) > 0:
                    action[self._IDX_SLOT_ARG] = int(valid_slots[0])
                else:
                    # Nothing to destroy -- revert to noop
                    action[self._IDX_FUNC]     = self._FUNC_NOOP
                    action[self._IDX_SLOT_ARG] = 0

        # 3. Clear craft arg unless crafting
        if func != 4:
            action[self._IDX_CRAFT_ARG] = 0

        # 4. Guarantee at least movement or a functional action
        moving = (
            action[self._IDX_FORWARD] != 0
            or action[self._IDX_STRAFE] != 0
            or action[self._IDX_JUMP]   != 0
        )
        acting = action[self._IDX_FUNC] != self._FUNC_NOOP
        if not moving and not acting:
            action[self._IDX_FORWARD] = 1  # force forward

        return action

    def _compute_reward(self, obsidian_mask: np.ndarray) -> float:
        current_count = int(obsidian_mask.sum())
        new_blocks = max(0, current_count - self._prev_obsidian_count)
        self._prev_obsidian_count = current_count

        reward = new_blocks * 1.0 - 0.01  # placement reward + time penalty

        # One-time portal frame bonus
        if not self._portal_bonus_given and _has_portal_frame(obsidian_mask.astype(bool)):
            reward += 5.0
            self._portal_bonus_given = True

        return reward

    # ------------------------------------------------------------------
    def reset(self, *, seed=None, options=None):
        raw_obs = self._env.reset()
        self._prev_obsidian_count = 0
        self._portal_bonus_given = False
        self._last_masks = raw_obs.get("masks", {})
        obs = self._extract_obs(raw_obs)
        return obs, {}

    def step(self, action):
        action = self.masked_action(np.asarray(action))
        raw_obs, _inner_reward, terminated, info = self._env.step(action)
        obs = self._extract_obs(raw_obs)
        reward = self._compute_reward(obs["obsidian_mask"])
        truncated = False
        return obs, reward, terminated, truncated, info

    def render(self):
        # Return the RGB image for logging / recording
        return self._env.render()

    def close(self):
        self._env.close()


In [5]:
class ObsidianExtractor(BaseFeaturesExtractor):
    """
    Encodes:
      rgb   → small CNN  → 256-d vector
      voxel → flatten + MLP → 64-d vector
    Concatenates → 320-d features
    """

    def __init__(self, observation_space: spaces.Dict, features_dim: int = 320):
        super().__init__(observation_space, features_dim)

        H, W = observation_space["rgb"].shape[1], observation_space["rgb"].shape[2]

        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute CNN output size dynamically
        dummy = torch.zeros(1, 3, H, W)
        cnn_out = self.cnn(dummy).shape[1]

        self.cnn_head = nn.Sequential(
            nn.Linear(cnn_out, 256),
            nn.ReLU(),
        )

        voxel_flat = int(np.prod(VOXEL_SHAPE))
        self.voxel_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(voxel_flat, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )

    def forward(self, observations: dict) -> torch.Tensor:
        rgb = observations["rgb"].float() / 255.0
        voxel = observations["obsidian_mask"]

        cnn_feat = self.cnn_head(self.cnn(rgb))
        vox_feat = self.voxel_head(voxel)

        return torch.cat([cnn_feat, vox_feat], dim=1)

In [6]:
def make_env():
    def _init():
        env = ObsidianPortalEnv(cfg=BASE_CFG)
        return env
    return _init

def train(
    total_timesteps: int = 1_000_000,
    n_envs: int = 1,           # increase if you have multiple GPU/CPU cores
    save_path: str = "./checkpoints",
    log_path: str = "./logs",
    record_video: bool = False,
    video_path: str = "./videos",
    video_freq: int = 10_000,  # record a clip every N steps
    video_length: int = 500,   # length of each clip in steps
):
    print("Building vectorised environment …")
    vec_env = DummyVecEnv([make_env() for _ in range(n_envs)])
    vec_env = VecMonitor(vec_env, log_path)

    if record_video:
        # VecVideoRecorder wraps the vec env and writes an MP4 every
        # video_freq steps.  render_mode must be "rgb_array" (set in
        # ObsidianPortalEnv.metadata).
        vec_env = VecVideoRecorder(
            vec_env,
            video_folder=video_path,
            record_video_trigger=lambda step: step % video_freq == 0,
            video_length=video_length,
            name_prefix="ppo_obsidian",
        )
        print(f"Video recording enabled → {video_path}  (every {video_freq:,} steps, {video_length} steps long)")

    policy_kwargs = dict(
        features_extractor_class=ObsidianExtractor,
        features_extractor_kwargs=dict(features_dim=320),
        net_arch=dict(pi=[256, 128], vf=[256, 128]),
    )

    model = PPO(
        policy="MultiInputPolicy",
        env=vec_env,
        learning_rate=3e-4,
        n_steps=512,           # rollout length per env
        batch_size=64,
        n_epochs=4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,         # encourage exploration
        vf_coef=0.5,
        max_grad_norm=0.5,
        tensorboard_log=log_path,
        verbose=1,
        policy_kwargs=policy_kwargs,
    )

    checkpoint_cb = CheckpointCallback(
        save_freq=50_000 // n_envs,
        save_path=save_path,
        name_prefix="ppo_obsidian",
    )

    print(f"Training PPO for {total_timesteps:,} timesteps …")
    model.learn(
        total_timesteps=total_timesteps,
        callback=checkpoint_cb,
        progress_bar=True,
    )

    model.save(f"{save_path}/ppo_obsidian_final")
    print("Done. Model saved.")
    return model


# ---------------------------------------------------------------------------
# Quick smoke-test (single rollout, no training)
# ---------------------------------------------------------------------------
def smoke_test(debug: bool = True):
    print("Running smoke test …")
    env = ObsidianPortalEnv(cfg=BASE_CFG)

    # Print raw MineDojo obs keys before wrapping, so we can verify the
    # correct path to voxel block names.
    if debug:
        raw_env = minedojo.make(**BASE_CFG)
        raw_obs = raw_env.reset()
        print("  Raw MineDojo obs keys:", list(raw_obs.keys()))
        if "voxels" in raw_obs:
            print("  raw_obs['voxels'] keys:", list(raw_obs["voxels"].keys()))
        elif "voxel" in raw_obs:
            print("  raw_obs['voxel'] keys:", list(raw_obs["voxel"].keys()))
        else:
            print("  WARNING: neither 'voxel' nor 'voxels' found in raw obs!")
            print("  All keys:", {k: type(v) for k, v in raw_obs.items()})
        raw_env.close()

    obs, _ = env.reset()
    print(f"  rgb shape        : {obs['rgb'].shape}")
    print(f"  obsidian_mask shape : {obs['obsidian_mask'].shape}")
    print(f"  obsidian voxels in initial obs: {obs['obsidian_mask'].sum():.0f}")

    total_reward = 0.0
    for step in range(10):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        print(f"  step {step:2d} | obsidian={obs['obsidian_mask'].sum():.0f} | reward={reward:.3f}")
        if terminated or truncated:
            break

    print(f"  Cumulative reward over smoke test: {total_reward:.3f}")
    env.close()
    print("Smoke test complete.")


In [7]:
# smoke_test()

In [8]:
train(total_timesteps=1_000_000, n_envs=4, save_path="./checkpoints", log_path="./logs", record_video=True, video_path="./videos", video_freq=50_000, video_length=500)

Building vectorised environment …


/home/jhu/miniconda3/envs/cs175/lib/python3.11/site-packages/numpy/core/numeric.py:330: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(a, fill_value, casting='unsafe')


AssertionError: The render_mode must be 'rgb_array', not None